**Преобразования признаков: теория, требования, когда применять и как инвертировать**

Цели ноутбука:

Дать чёткие определения и требования к преобразованиям признаков (домен, непрерывность, обратимость).

Покрыть часто используемые специальные преобразования: log, log1p, Box–Cox, Yeo–Johnson, power, sqrt, quantile, winsorize/clip, binning/one-hot, cyclical (sin/cos), arcsin‑sqrt для долей.

Показать индикаторную функцию I{·}, примеры чтения формул и мягкие (дифференцируемые) аппроксимации индикаторов.



**Ключевые определения и требования**

Преобразование f: X → Z применяют к признаку x для изменения его распределения/свойств.

Домeн (domain): некоторые преобразования требуют x > 0 (log, Box–Cox); Yeo–Johnson поддерживает отрицательные.

Непрерывность / дифференцируемость: важно для градиентных методов (нейросети, градиентный бустинг с градиентной оптимизацией).
Деревья не требуют гладкости.
Обратимость (invertibility): если нужно интерпретировать предсказания в оригинальной шкале, используйте обратимые преобразования и сохраняйте параметры.

Числовая стабильность: используйте np.log1p / np.expm1, избегайте деления на ноль и работы с бесконечностями.

Потеря информации: clip / binning / one-hot часто неинвертируемы (нельзя вернуть точное исходное значение).


**Индикаторная функция (indicator function)**

Нотация: $1_{A}(x)$ или I{A}(x). Определение: 1_{A}(x) = { 1, если x ∈ A; 0, иначе }.

Примеры использования:

One‑hot для категорий: c ∈ {A,B,C} → I{c=A}, I{c=B}, I{c=C}.

Биннинг: b_k(x) = I{a_{k-1} < x ≤ a_k}.

Piecewise/кусо‑линейная модель: f(x) = β0 + β1·x·I{x≤t} + β2·x·I{x>t}.


Важность: индикаторы задают разрывы в модели (непрерывность нарушается).
Для градиентных методов часто предпочтительны мягкие (дифференцируемые) аппроксимации:

sigmoid(k(x − t)) ≈ I{x > t} для больших k,

tanh(k(x − t)) аналогично,

softplus = ln(1 + exp(x)) — гладкая версия ReLU.


**Частые преобразования — краткая шпаргалка (что, формула, домен, обратное)**

Log:
y = ln(x), домен x > 0. inverse: x = exp(y).

Log1p:
y = ln(1 + x), домен x ≥ 0 (удобно при нулях). inverse: x = exp(y) − 1 (np.expm1).

Power transforms — Box–Cox:
T(x; λ) = (x^λ − 1) / λ, λ ≠ 0; T(x; 0) = ln(x). Требует x > 0.
inverse: x = (λ·T + 1)^(1/λ).

Yeo–Johnson:
Поддерживает x ∈ ℝ (включая отрицательные). Реализован в sklearn.preprocessing.PowerTransformer(method='yeo-johnson'). Имеет обратное.

Sqrt:
y = sqrt(x), домен x ≥ 0. inverse: x = y^2.

Reciprocal / inverse:
y = 1 / x, домен x ≠ 0, чувствительно к шуму.

QuantileTransformer:
Преобразует эмпирический CDF в uniform или normal. inverse – аппроксимативная (эмпирические квантили).

Winsorize / clip:
x' = clip(x, lower, upper) или winsorize. Потеря информации → неинвертируемы.

Binning + one-hot:
Урезают/категоризируют значение. Обычно неинвертируемы.

Cyclical:
Для часов/дней/углов: sin(2π·x / period), cos(2π·x / period). Обратное через atan2 (фаза).

Arcsin‑sqrt:
Для долей p ∈ [0,1]: y = arcsin(sqrt(p)), inverse: p = sin(y)^2. Уменьшает вариацию при долях близких к 0/1.

In [ ]:
# Выполните эту ячейку
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')
from scipy import stats
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle

**Log и Log1p — когда и зачем**

Смысл: уменьшение правой скошенности (skewness), уменьшение влияния больших значений (выбросов).

Требования: log(x) требует x > 0.
Если есть 0 или очень маленькие положительные — используйте log1p (ln(1+x)) или сдвиг x_shift = x + c (c > 0).

Практика: для счётчиков (counts) обычно log1p или sqrt. Для цен часто log.

Обратное: exp / expm1.

Числовая стабильность: используйте np.log1p и np.expm1.

Примечание по интерпретации предсказаний:


Если модель обучена на y = log(target) и вы хотите предсказывать в исходном масштабе, то простая инверсия exp(y_pred) даёт медиану (в случае лог‑нормального шума) не обязательно ожидаемое значение. Для коррекции смещения используйте оценки дисперсии остатков или метод Duan (smearing estimator).


In [ ]:
# Синтетические данные для демонстрации
rng = np.random.RandomState(0)
x = np.concatenate([rng.exponential(scale=50, size=900), rng.exponential(scale=300, size=100)])
x = np.maximum(0, x)  # гарантируем >=0

# Log1p
x_log1p = np.log1p(x)
# inverse
x_back = np.expm1(x_log1p)

print("Max abs diff back:", np.max(np.abs(x - x_back)))

# График
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(x, kde=True).set_title('orig (skew)')
plt.subplot(1,2,2)
sns.histplot(x_log1p, kde=True).set_title('log1p(x)')
plt.tight_layout()

**Power‑transforms: Box–Cox и Yeo–Johnson**

Box–Cox:

Формула: T(x; λ) = (x^λ − 1) / λ, для λ ≠ 0; при λ = 0 → ln(x).

Требует x > 0. Параметр λ подбирают методом максимального правдоподобия (реализация в scipy.stats.boxcox).
Обратно: x = (λ·T + 1)^(1/λ).
Применяется для стабилизации дисперсии и приближения к нормальному распределению.

Yeo–Johnson:
Модификация, работающая для x ≥ 0 и x < 0.

Формула чуть сложнее (см. презентацию к занятию). Доступен в sklearn.preprocessing.PowerTransformer(method='yeo-johnson').
Имеет inverse_transform в sklearn.
Когда выбирать:

Если все значения > 0 и хочется параметрически найти лучшую степень — Box–Cox.
Если есть отрицательные значения — Yeo–Johnson.

In [ ]:
def inv_boxcox_safe(z, lmbda, tol=1e-8):

    z = np.asarray(z)
    if np.isclose(lmbda, 0.0, atol=tol):
        return np.exp(z)
    else:
        return np.power(lmbda * z + 1.0, 1.0 / lmbda)


try:
    from scipy.special import inv_boxcox as sp_inv_boxcox
    def inv_boxcox_wrapper(z, lmbda):
        try:
            return sp_inv_boxcox(z, lmbda)
        except Exception:
            return inv_boxcox_safe(z, lmbda)
    inv_boxcox = inv_boxcox_wrapper
except Exception:
    inv_boxcox = inv_boxcox_safe

# Подготовим положительный признак
rng = np.random.RandomState(0)
x_pos = rng.exponential(scale=50, size=1000) + 1e-6  # >0

# Box-Cox
x_boxcox, lmbda = stats.boxcox(x_pos)  # подобран λ
x_boxcox_back = inv_boxcox(x_boxcox, lmbda)

print("λ (Box-Cox):", lmbda)
print("Max abs diff inv:", np.max(np.abs(x_pos - x_boxcox_back)))

# Yeo-Johnson на данных с отрицаниями
from sklearn.preprocessing import PowerTransformer
x_neg = rng.normal(loc=0, scale=10, size=1000)
pt = PowerTransformer(method='yeo-johnson')
x_yj = pt.fit_transform(x_neg.reshape(-1,1)).ravel()
x_yj_back = pt.inverse_transform(x_yj.reshape(-1,1)).ravel()
print("Max abs diff Yeo-Johnson inv:", np.max(np.abs(x_neg - x_yj_back)))

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); sns.histplot(x_pos, kde=True).set_title('orig positive')
plt.subplot(1,2,2); sns.histplot(x_boxcox, kde=True).set_title(f'boxcox λ={lmbda:.3f}')
plt.tight_layout()
plt.show()

In [ ]:
sns.set(style='whitegrid')
rng = np.random.RandomState(0)

# Данные
x_pos = rng.exponential(scale=50, size=1000) + 1e-6
x_boxcox, lmbda = stats.boxcox(x_pos)

x_neg = rng.normal(loc=0, scale=10, size=1000)

# PowerTransformer по умолчанию standardize=True (Yeo-Johnson + z-score)
pt = PowerTransformer(method='yeo-johnson', standardize=True)
x_yj = pt.fit_transform(x_neg.reshape(-1,1)).ravel()

# Если хотим увидеть только степенную трансформацию без стандартизации:
pt_nostd = PowerTransformer(method='yeo-johnson', standardize=False)
x_yj_nostd = pt_nostd.fit_transform(x_neg.reshape(-1,1)).ravel()

# Визуализация: 4 панели
plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
sns.histplot(x_pos, kde=True)
plt.title('orig positive')

plt.subplot(2,2,2)
sns.histplot(x_boxcox, kde=True)
plt.title(f'Box-Cox (λ={lmbda:.3f})')

plt.subplot(2,2,3)
sns.histplot(x_neg, kde=True)
plt.title('orig (may be negative)')

plt.subplot(2,2,4)
sns.histplot(x_yj, kde=True)
plt.title('Yeo-Johnson (standardized)')

plt.tight_layout()
plt.show()

# "не стандартизированный" Yeo-Johnson
plt.figure(figsize=(6,4))
sns.histplot(x_yj_nostd, kde=True)
plt.title('Yeo-Johnson (standardize=False)')
plt.show()

**QuantileTransformer (rank‑based)**


Что делает: переводит эмпирические значения в квантили и отображает их в равномерное или нормальное распределение. Убирает сильную скошенность, делает распределение близким к Normal или Uniform.

Преимущества: мощное выравнивание распределения.

Недостатки: преобразование не параметрическое → inverse_transform аппроксимативна (опирается на эмпирические квантили). При новых (вне обучающей выборки) значениях и при небольших данных инверсия и экстраполяция могут быть плохими.

Когда использовать: когда важна нормализация распределения для моделей, чувствительных к форме распределения (например, линейная регрессия, некоторые методы оптимизации).

In [ ]:
from sklearn.preprocessing import QuantileTransformer

x = np.concatenate([rng.exponential(scale=50, size=950), rng.exponential(scale=300, size=50)])
qt = QuantileTransformer(output_distribution='normal', random_state=0)
x_qt = qt.fit_transform(x.reshape(-1,1)).ravel()
x_qt_back = qt.inverse_transform(x_qt.reshape(-1,1)).ravel()

print("Max abs diff quantile inverse (approx):", np.max(np.abs(x - x_qt_back)))

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); sns.histplot(x, kde=True).set_title('orig (skew)')
plt.subplot(1,2,2); sns.histplot(x_qt, kde=True).set_title('quantile -> normal')
plt.tight_layout()

**Winsorize / Clip — обработка выбросов**


Clip: x' = clip(x, lower, upper) — заменяет значения ниже/выше порогов.

Неинвертируемо (потеря информации).

Winsorize: замена крайних значений на значение квантили, но сохраняется порядок внутри отрезка. Чаще используют для робастной статистики.

Когда: когда выбросы являются артефактами или несущественны для модели; но будьте осторожны — потеря информации может ухудшить важные наблюдения.


In [ ]:
x = np.concatenate([rng.normal(50, 5, size=990), rng.normal(200, 20, size=10)])
low, high = np.percentile(x, [1, 99])
x_clip = np.clip(x, low, high)
x_wins = winsorize(x, limits=(0.01, 0.01))

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); sns.histplot(x, kde=True).set_title('orig with outliers')
plt.subplot(1,3,2); sns.histplot(x_clip, kde=True).set_title('clip 1%-99%')
plt.subplot(1,3,3); sns.histplot(x_wins, kde=True).set_title('winsorize 1%')
plt.tight_layout()

**Биннинг и one‑hot**

Биннинг: перевод непрерывного признака в категории по порогам: bin_k(x) = I{a_{k-1} < x ≤ a_k}.

One‑hot: для каждой категории создаётся индикатор. Полезно для деревьев и линейных моделей (с фиксацией reference level).

Обратимость: утрачивается точная информация; можно восстановить только номер бина.

Практика: используйте quantile‑бины (равные по числу наблюдений) или domain‑specific bins.


In [ ]:
s = np.random.exponential(50, size=200)
bins = [0, 25, 75, np.inf]
labels = ['low', 'mid', 'high']
s_bins = pd.cut(s, bins=bins, labels=labels)
pd.get_dummies(s_bins, prefix='bin').head()

**Cyclical (час/месяц/угол) — sin/cos кодирование**

Для признаков с циклической природой (час, день, месяц, направление ветра) используйте два признака:

sin = sin(2π·x / period)

cos = cos(2π·x / period)

Преимущество: соседние значения имеют близкие представления, сохраняется цикличность.

Обратное преобразование: фаза φ = atan2(sin, cos), затем x = (period * φ) / (2π) (с поправкой на диапазон).

In [ ]:
hours = np.arange(0, 24)
sin_h = np.sin(2*np.pi*hours/24)
cos_h = np.cos(2*np.pi*hours/24)

plt.figure(figsize=(8,3))
plt.subplot(1,2,1); plt.plot(hours, sin_h, marker='o'); plt.title('sin(hour)')
plt.subplot(1,2,2); plt.plot(hours, cos_h, marker='o'); plt.title('cos(hour)')
plt.tight_layout()

# inverse example for a random hour
h0 = 5
s, c = np.sin(2*np.pi*h0/24), np.cos(2*np.pi*h0/24)
phi = np.arctan2(s, c)  # in radians [-pi, pi]
hour_recon = (phi % (2*np.pi)) * 24 / (2*np.pi)
print(h0, "-> sin,cos -> reconstructed hour:", hour_recon)

**Мягкие индикаторы и сплайны**

Soft indicator: s_k(x) = sigmoid(k (x − t_k)).
При k → ∞ стремится к жёсткому индикатору. Полезно при обучении градиентными методами.

Сплайны (spline) позволяют моделировать кусочно‑полиномиальные функции с заданной гладкостью. sklearn.preprocessing.SplineTransformer может быть полезен.

Когда применять: если требуется гибкая, но гладкая модель кусочно‑линейной/полиномиальной зависимости.

In [ ]:
x = np.linspace(-3, 3, 400)
t = 0.5
k_list = [1, 5, 20]
plt.figure(figsize=(8,5))
plt.plot(x, (x>t).astype(float), label='hard indicator 1_{x>t}')
for k in k_list:
    s = 1.0 / (1.0 + np.exp(-k*(x - t)))
    plt.plot(x, s, label=f'sigmoid k={k}')
plt.ylim(-0.1, 1.1)
plt.legend()
plt.title('Hard indicator vs soft (sigmoid) approximations')
plt.xlabel('x')
plt.show()

**Обратное преобразование целевой переменной и коррекция смещения**


Проблема: если модель обучалась на y = ln(target), и мы инвертируем предсказание через exp(ŷ), то в общем случае E[exp(ŷ)] ≠ exp(E[ŷ]) (т.к. exp — выпуклая функция). Для модели с аддитивным нормальным шумом на лог‑шкале (log‑normal noise):


Пусть ln(Y) = μ + ε, ε ∼ N(0, σ^2). Тогда E[Y] = exp(μ + σ^2 / 2).
Итого: если вы прогнозируете μ̂ = E[ln Y | X], корректная оценка для Y может быть exp(μ̂ + 0.5 σ̂^2) при условии нормальности остатка.

Практика:

Простейший подход: корректировать сдвигом 0.5 * variance_of_residuals_on_log_scale.

Более точный (без предположения нормальности):
Duan's smearing estimator:
Обучаем модель на ln(Y). Получаем остатки r_i = ln(y_i) − μ̂_i.
Smearing factor s = mean(exp(r_i)).

Тогда скорректированное предсказание: ŷ = exp(μ̂_new) * s.
Смearing корректно аппроксимирует E[exp(ε)] без предположения о нормальности, но предполагает, что residuals распределение стационарно между train и test.


In [ ]:
rng = np.random.RandomState(1)
X = rng.normal(size=(500,1))
eps = rng.normal(scale=0.4, size=500)  # log‑шум
y = np.exp(2*X.ravel() + eps)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

model = LinearRegression()
y_train_log = np.log(y_train)
model.fit(X_train, y_train_log)
mu_hat_test = model.predict(X_test)

# Прямая инверсия (простая exp)
y_pred_direct = np.exp(mu_hat_test)

# Duan smearing
r_train = y_train_log - model.predict(X_train)
s_smear = np.mean(np.exp(r_train))
y_pred_smear = np.exp(mu_hat_test) * s_smear

# Нормальная корректировка (если считать остатки нормальными)
sigma2 = np.var(r_train, ddof=0)
y_pred_normal_corr = np.exp(mu_hat_test + 0.5 * sigma2)

# RMSE через sqrt(MSE) — работает с любой версией sklearn
rmse_direct = np.sqrt(mean_squared_error(y_test, y_pred_direct))
rmse_smear = np.sqrt(mean_squared_error(y_test, y_pred_smear))
rmse_normal = np.sqrt(mean_squared_error(y_test, y_pred_normal_corr))

print("RMSE direct:         ", rmse_direct)
print("RMSE Duan smearing:  ", rmse_smear)
print("RMSE normal-corrected:", rmse_normal)

**Сохранение трансформеров и пайплайнов**

Всегда сохраняйте параметры преобразований (λ для Box–Cox, quantiles, min/max, параметры PowerTransformer и т.д.) — иначе inverse_transform на продакшене будет некорректной.

Часто используют sklearn Pipeline и сериализуют его: pickle / joblib.

Пример:
pipeline = Pipeline([('pt', PowerTransformer()), ('scaler', StandardScaler())])
pickle.dump(pipeline, open('pipeline.pkl', 'wb'))
pipeline = pickle.load(open('pipeline.pkl', 'rb'))


In [ ]:
pt = PowerTransformer(method='yeo-johnson')
data = np.random.normal(size=(100,1))
pt.fit(data)
# Сохранить
with open('power_transformer.pkl', 'wb') as f:
    pickle.dump(pt, f)
# Загрузить
with open('power_transformer.pkl', 'rb') as f:
    pt2 = pickle.load(f)
# Проверка
x = np.array([[1.0]])
print("Original transform:", pt.transform(x))
print("Loaded transform:", pt2.transform(x))

**Практические советы**

Для признаков с нулями/положительными: log1p часто безопаснее, чем log с сдвигом.

Для признаков с отрицательными: Yeo–Johnson.

Если важна обратимость и предсказания нужны в исходном масштабе — убедитесь, что transform имеет корректный inverse и вы сохраняете параметры.

Для градиентных и нейронных моделей предпочитайте гладкие трансформации или мягкие индикаторы; для деревьев — binning и one‑hot часто работают нормально.

Для целевой переменной: при лог‑трансформации используйте smearing или корректировку через дисперсию, если вам нужен unbiased estimate среднего.

Для quantile transforms: inverse_transform аппроксимативна — осторожно при применении на новых данных.

Всегда визуализируйте: histplot до/после, QQ‑plot, scatter target vs feature (до/после).


**StandardScaler (z-нормализация)**
— это приведение значений признаков к единому масштабу, чаще всего в фиксированный диапазон (например,
[0,1]
[0,1]).
Цель:
сделать признаки сопоставимыми по масштабу, чтобы ни один из них не доминировал в модели из-за больших числовых значений. x_scaled = (x - mean) / std


In [ ]:
rng = np.random.RandomState(42)
X = rng.normal(loc=100, scale=20, size=(1000, 1))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_back = scaler.inverse_transform(X_scaled)

print("Mean after scaling:", X_scaled.mean())
print("Std after scaling:", X_scaled.std())
print("Max abs diff inverse:", np.max(np.abs(X - X_back)))

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(X, kde=True).set_title("original")

plt.subplot(1,2,2)
sns.histplot(X_scaled, kde=True).set_title("StandardScaler")
plt.tight_layout()
plt.show()

**MinMaxScaler (шкалирование в [0,1])** - это преобразование признаков таким образом, чтобы изменить их масштаб, сохранив форму распределения.
Часто используется:
приведение среднего к 0
приведение стандартного отклонения к 1: x_mm = (x - min) / (max - min)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

X = rng.normal(loc=50, scale=15, size=(1000, 1))

mm = MinMaxScaler(feature_range=(0, 1))
X_mm = mm.fit_transform(X)
X_mm_back = mm.inverse_transform(X_mm)

print("Min after scaling:", X_mm.min())
print("Max after scaling:", X_mm.max())
print("Max abs diff inverse:", np.max(np.abs(X - X_mm_back)))

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(X, kde=True).set_title("original")

plt.subplot(1,2,2)
sns.histplot(X_mm, kde=True).set_title("MinMaxScaler [0,1]")
plt.tight_layout()
plt.show()


**RobustScaler (устойчив к выбросам)** - это метод масштабирования данных, который использует медиану и межквартильный размах (IQR) вместо среднего и стандартного отклонения.

x_rb = (x - median) / IQR

IQR = Q3 - Q1

    median — медиана
    Q1 — 25-й перцентиль
    Q3 — 75-й перцентиль

Центрирует данные относительно медианы
Делит на межквартильный размах
Менее чувствителен к выбросам, чем StandardScaler

In [ ]:
from sklearn.preprocessing import RobustScaler

X = np.concatenate([
    rng.normal(0, 1, size=990),
    rng.normal(15, 5, size=10)
]).reshape(-1,1)

rb = RobustScaler()
X_rb = rb.fit_transform(X)

print("Median after scaling (≈0):", np.median(X_rb))

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(X, kde=True).set_title("original with outliers")

plt.subplot(1,2,2)
sns.histplot(X_rb, kde=True).set_title("RobustScaler")
plt.tight_layout()
plt.show()
